In [48]:
from google.colab import drive

drive.mount("/content/gdrive")

!pip install torch numpy requests scikit-learn matplotlib

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [49]:
import torch
import torch.nn as nn
import numpy as np
import requests
import json
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from pathlib import Path

In [50]:
def fetch_prices(token_id: str, days: int) -> list[float]:
    """Fetch daily closing prices from CoinGecko."""
    url = f"https://api.coingecko.com/api/v3/coins/{token_id}/market_chart"
    params = {"vs_currency": "usd", "days": days, "interval": "daily"}

    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()  # raise exception on 4xx/5xx

    data = response.json()

    # extract from [timestamp, x]
    prices = [point[1] for point in data["prices"]]
    market_caps = [point[1] for point in data["market_caps"]]
    volumes = [point[1] for point in data["total_volumes"]]

    return prices, market_caps, volumes
print(fetch_prices("ethereum", 3))

([1637.8350754267938, 1621.2977067034271, 1671.88909163853, 1665.8023929993435], [197708727281.66705, 195685388741.0592, 201775713182.1136, 201048268652.21692], [14127918160.208885, 13051388908.792965, 11763657711.167143, 10128560082.29436])


In [51]:
def create_sequences(data, lookback):
    X = []
    y = []

    for i in range(len(data[0]) - lookback):
        input_seq = []
        for j in range(i, i + lookback):
            input_seq.append([data[0][j],data[1][j],data[2][j]])
        X.append(input_seq)
        y.append(data[0][i+lookback])

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

In [52]:
token = "ethereum"
days = 365
lookback = 7

prices, market_caps, volumes = fetch_prices(token, days)

prices_np = np.array(prices, dtype=np.float32)
market_caps_np = np.array(market_caps, dtype=np.float32)
volumes_np = np.array(volumes, dtype=np.float32)


#calculate min , max of all 3
feature_min = [prices_np.min(),market_caps_np.min(),volumes_np.min()]
feature_max = [prices_np.max(),market_caps_np.max(),volumes_np.max()]


for i in range(3):
  normalized_price = (prices_np - feature_min[0]) / (feature_max[0] - feature_min[0])
  normalized_market_caps = (market_caps_np - feature_min[1]) / (feature_max[1] - feature_min[1])
  normalized_volumes = (volumes_np - feature_min[2]) / (feature_max[2] - feature_min[2])

features = [
    normalized_price,
    normalized_market_caps,
    normalized_volumes
]

X, y = create_sequences(features, lookback)

print("X shape:", X.shape)
print("y shape:", y.shape)

#prepare tensors (reshape to (a,b,c) and (c,d))
X_tensor = torch.tensor(X, dtype=torch.float32).reshape(len(X), lookback, 3)
y_tensor = torch.tensor(y, dtype=torch.float32).reshape(len(y), 1)

print("X_tensor shape:", X_tensor.shape)
print("y_tensor shape:", y_tensor.shape)

X shape: (359, 7, 3)
y shape: (359,)
X_tensor shape: torch.Size([359, 7, 3])
y_tensor shape: torch.Size([359, 1])


In [53]:
#split for training and test
train_size = int(0.8 * len(X_tensor))

X_train = X_tensor[:train_size]
y_train = y_tensor[:train_size]

X_test = X_tensor[train_size:]
y_test = y_tensor[train_size:]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: torch.Size([287, 7, 3])
y_train: torch.Size([287, 1])
X_test: torch.Size([72, 7, 3])
y_test: torch.Size([72, 1])


In [54]:
#training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train = X_train.to(device)
y_train = y_train.to(device)
X_test = X_test.to(device)
y_test = y_test.to(device)

print("Using device:", device)

Using device: cuda


In [55]:
class PriceLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, (hidden, cell) = self.lstm(x)

        last_time_step = out[:, -1, :]

        prediction = self.fc(last_time_step)

        return prediction

In [56]:
input_size = 3
hidden_size = 128
num_layers = 4

model = PriceLSTM(input_size, hidden_size, num_layers).to(device)

loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

print(model)

epochs = 300

for epoch in range(epochs):
  model.train()

  #forward in def actuallya takes input parameters to that function
  predictions = model(X_train)

  #calculate loss
  loss = loss_fn(predictions, y_train)

  # I dont get a fuck
  # clear old gradients
  optimizer.zero_grad()

    # calculate new gradients
  loss.backward()

    # update model weights
  optimizer.step()

  if (epoch + 1) % 10 == 0:
          model.eval()

          with torch.no_grad():
              test_predictions = model(X_test)
              test_loss = loss_fn(test_predictions, y_test)

          print(
              f"Epoch {epoch + 1}/{epochs} | "
              f"Train Loss: {loss.item():.6f} | "
              f"Test Loss: {test_loss.item():.6f}"
          )


PriceLSTM(
  (lstm): LSTM(3, 128, num_layers=4, batch_first=True)
  (fc): Linear(in_features=128, out_features=1, bias=True)
)
Epoch 10/300 | Train Loss: 0.065128 | Test Loss: 0.166578
Epoch 20/300 | Train Loss: 0.012459 | Test Loss: 0.003859
Epoch 30/300 | Train Loss: 0.008999 | Test Loss: 0.003261
Epoch 40/300 | Train Loss: 0.007135 | Test Loss: 0.005449
Epoch 50/300 | Train Loss: 0.005435 | Test Loss: 0.001099
Epoch 60/300 | Train Loss: 0.004805 | Test Loss: 0.001082
Epoch 70/300 | Train Loss: 0.004258 | Test Loss: 0.001031
Epoch 80/300 | Train Loss: 0.003850 | Test Loss: 0.001111
Epoch 90/300 | Train Loss: 0.003320 | Test Loss: 0.000921
Epoch 100/300 | Train Loss: 0.004742 | Test Loss: 0.000968
Epoch 110/300 | Train Loss: 0.003392 | Test Loss: 0.000796
Epoch 120/300 | Train Loss: 0.003091 | Test Loss: 0.000948
Epoch 130/300 | Train Loss: 0.002573 | Test Loss: 0.000787
Epoch 140/300 | Train Loss: 0.002219 | Test Loss: 0.000741
Epoch 150/300 | Train Loss: 0.002037 | Test Loss: 0.0006

In [57]:
model.eval()

with torch.no_grad():
    test_predictions = model(X_test)

# move tensors back to CPU and convert to numpy
test_predictions_norm = test_predictions.cpu().numpy()
actual_values_norm = y_test.cpu().numpy()

# normalized MAE
normalized_errors = np.abs(test_predictions_norm - actual_values_norm)
normalized_mae = normalized_errors.mean()

# inverse scale back to real prices
test_predictions = test_predictions_norm * (feature_max[0] - feature_min[0]) + feature_min[0]
actual_values = actual_values_norm * (feature_max[0] - feature_min[0]) + feature_min[0]

# real dollar MAE
errors = np.abs(test_predictions - actual_values)
mae = errors.mean()

rmse = np.sqrt(((test_predictions - actual_values) ** 2).mean())

for a in range(len(test_predictions)):
    print(
        f"predicted is {test_predictions[a][0]:.2f}, "
        f"real is {actual_values[a][0]:.2f}, "
        f"error is {errors[a][0]:.2f}"
    )

print("MAE:", mae)
print("RMSE:", rmse)
print("Normalized MAE:", normalized_mae)

predicted is 2148.96, real is 2056.89, error is 92.07
predicted is 2098.49, real is 2053.61, error is 44.88
predicted is 2076.03, real is 2064.99, error is 11.04
predicted is 2085.86, real is 2109.01, error is 23.14
predicted is 2125.31, real is 2107.83, error is 17.48
predicted is 2140.50, real is 2241.82, error is 101.32
predicted is 2228.52, real is 2190.48, error is 38.05
predicted is 2220.72, real is 2188.97, error is 31.75
predicted is 2201.90, real is 2245.05, error is 43.14
predicted is 2236.24, real is 2285.47, error is 49.23
predicted is 2285.24, real is 2192.16, error is 93.08
predicted is 2231.53, real is 2371.86, error is 140.34
predicted is 2334.95, real is 2323.22, error is 11.73
predicted is 2344.87, real is 2359.68, error is 14.81
predicted is 2356.53, real is 2348.70, error is 7.82
predicted is 2347.84, real is 2421.01, error is 73.17
predicted is 2400.45, real is 2350.94, error is 49.52
predicted is 2372.83, real is 2264.81, error is 108.02
predicted is 2285.56, real

In [58]:
import os

save_dir = "/content/gdrive/MyDrive/learn_models"
os.makedirs(save_dir, exist_ok=True)

save_path = f"{save_dir}/{token}_lstm.pt"

checkpoint = {
    "model_state_dict": model.state_dict(),

    "token": token,
    "days": days,
    "lookback": lookback,

    "input_size": input_size,
    "hidden_size": hidden_size,
    "num_layers": num_layers,

    "feature_min": [float(x) for x in feature_min],
    "feature_max": [float(x) for x in feature_max],
    "features": ["price", "market_cap", "volume"],
    "target": "next_day_price"
}

torch.save(checkpoint, save_path)

print("saved model to:", save_path)

saved model to: /content/gdrive/MyDrive/learn_models/ethereum_lstm.pt
